In [1]:
import pandas as pd
import numpy as np
import pickle

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
# Content Based

with open(
    r"E:\Reel-Recommendation-Engine\models\content_similarity.pkl",
    "rb"
) as f:
    content_similarity = pickle.load(f)

with open(
    r"E:\Reel-Recommendation-Engine\models\video_indices.pkl",
    "rb"
) as f:
    video_indices = pickle.load(f)

with open(
    r"E:\Reel-Recommendation-Engine\models\tfidf_vectorizer.pkl",
    "rb"
) as f:
    tfidf_vectorizer = pickle.load(f)

print("Content Models Loaded")

Content Models Loaded


In [3]:
with open(
    r"E:\Reel-Recommendation-Engine\models\item_similarity.pkl",
    "rb"
) as f:
    item_similarity = pickle.load(f)

with open(
    r"E:\Reel-Recommendation-Engine\models\video_to_idx.pkl",
    "rb"
) as f:
    video_to_idx = pickle.load(f)

with open(
    r"E:\Reel-Recommendation-Engine\models\idx_to_video.pkl",
    "rb"
) as f:
    idx_to_video = pickle.load(f)

print("Collaborative Models Loaded")

Collaborative Models Loaded


In [4]:
with open(
    r"E:\Reel-Recommendation-Engine\models\svd_model.pkl",
    "rb"
) as f:
    svd_model = pickle.load(f)

with open(
    r"E:\Reel-Recommendation-Engine\models\all_videos.pkl",
    "rb"
) as f:
    all_videos = pickle.load(f)

print("SVD Model Loaded")

SVD Model Loaded


In [5]:
videos_df = pd.read_csv(
    r"E:\Reel-Recommendation-Engine\data\processed\videos_cleaned.csv"
)

users_df = pd.read_csv(
    r"E:\Reel-Recommendation-Engine\data\processed\users_cleaned.csv"
)

interactions_df = pd.read_csv(
    r"E:\Reel-Recommendation-Engine\data\processed\interactions_cleaned.csv"
)

print(videos_df.shape)
print(users_df.shape)
print(interactions_df.shape)

(7583, 12)
(27285, 31)
(1414622, 22)


In [6]:
sample_video = videos_df["video_id"].iloc[0]

print("Testing Video:", sample_video)

idx = video_indices[sample_video]

similarity_scores = list(
    enumerate(content_similarity[idx])
)

similarity_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

top_videos = similarity_scores[1:11]

recommendations = []

for video_idx, score in top_videos:

    recommendations.append(
        (
            videos_df.iloc[video_idx]["video_id"],
            score
        )
    )

recommendations

Testing Video: 0


[(4121, 0.13904637954655436),
 (655, 0.09275468025508411),
 (3234, 0.09275468025508411),
 (2894, 0.08919214789888164),
 (481, 0.0858727873534052),
 (3174, 0.0858727873534052),
 (817, 0.06998347318804668),
 (103, 0.06977164259598541),
 (892, 0.06977164259598541),
 (935, 0.06977164259598541)]

In [7]:
sample_video = videos_df["video_id"].iloc[0]

video_index = video_to_idx[sample_video]

scores = list(
    enumerate(item_similarity[video_index])
)

scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

recommendations = []

for idx, score in scores[1:11]:

    recommendations.append(
        (
            idx_to_video[idx],
            score
        )
    )

recommendations

[(3409, 0.2912889412538552),
 (7580, 0.28422366156530615),
 (4183, 0.24164156691700736),
 (5508, 0.22999941779976985),
 (2374, 0.20407634175293504),
 (4944, 0.19918291753685743),
 (5153, 0.19744692833584934),
 (6503, 0.18487545736210995),
 (1172, 0.1783307444310914),
 (2194, 0.17606511391828972)]

In [8]:
sample_user = interactions_df["user_id"].iloc[0]

predictions = []

for video_id in all_videos[:100]:

    score = svd_model.predict(
        sample_user,
        video_id
    ).est

    predictions.append(
        (
            video_id,
            score
        )
    )

predictions = sorted(
    predictions,
    key=lambda x: x[1],
    reverse=True
)

predictions[:10]

[(1868, 0.9640118620550967),
 (2303, 0.6210722047389368),
 (4955, 0.49580639054625736),
 (4636, 0.4133953463507144),
 (5522, 0.23959319885149147),
 (3831, 0.22741911097457065),
 (597, 0.1953700207013894),
 (1164, 0.19188110799841726),
 (7438, 0.19130465379876138),
 (3706, 0.19099961854068656)]

In [9]:
def hybrid_recommendation(
    user_id,
    video_id,
    top_n=10
):

    content_idx = video_indices[video_id]

    content_scores = list(
        enumerate(
            content_similarity[content_idx]
        )
    )

    content_scores = sorted(
        content_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:51]

    hybrid_scores = []

    for idx, content_score in content_scores:

        candidate_video = videos_df.iloc[idx]["video_id"]

        svd_score = svd_model.predict(
            user_id,
            candidate_video
        ).est

        hybrid_score = (
            0.4 * content_score
            +
            0.6 * svd_score
        )

        hybrid_scores.append(
            (
                candidate_video,
                hybrid_score
            )
        )

    hybrid_scores = sorted(
        hybrid_scores,
        key=lambda x: x[1],
        reverse=True
    )

    return hybrid_scores[:top_n]

In [10]:
sample_user = interactions_df["user_id"].iloc[0]

sample_video = interactions_df["video_id"].iloc[0]

hybrid_recommendation(
    sample_user,
    sample_video
)

[(6400, 0.16575355906011174),
 (4121, 0.14935771321483596),
 (2640, 0.12307470243941325),
 (1211, 0.11734755992011418),
 (3136, 0.09580679270836748),
 (6681, 0.09067877275965353),
 (71, 0.07935309400097795),
 (7230, 0.07171513688109393),
 (6497, 0.07005851266274712),
 (5782, 0.06888713380593657)]

In [11]:
test_results = hybrid_recommendation(
    sample_user,
    sample_video,
    top_n=20
)

api_test_df = pd.DataFrame(
    test_results,
    columns=[
        "video_id",
        "score"
    ]
)

api_test_df.to_csv(
    r"E:\Reel-Recommendation-Engine\data\processed\api_test_results.csv",
    index=False
)

print("API Testing Completed")

API Testing Completed
